In [1]:
import pandas as pd
import numpy as np
import torch
from torch import nn
import re
from transformers import AutoTokenizer, AutoModel, Trainer, TrainingArguments, EarlyStoppingCallback
from sklearn.utils.class_weight import compute_class_weight
from sklearn.metrics import f1_score
from datasets import Dataset
from transformers import DataCollatorWithPadding
import os

# 1. CẤU HÌNH CƠ BẢN
MODEL_NAME = "../model/marbert_base"
STANCE2ID = {"Against": 0, "Favor": 1, "None": 2}
SENTIMENT2ID = {"Negative": 0, "Neutral": 1, "Positive": 2}
SARCASM2ID = {"No": 0, "Yes": 1}

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
data_collator = DataCollatorWithPadding(tokenizer=tokenizer)

# 2. TIỀN XỬ LÝ VÀ LOAD DATA CHUNG
def clean_arabic_tweet(text):
    if not isinstance(text, str): return ""
    text = re.sub(r"http\S+|www\.\S+", "", text)
    text = re.sub(r"@\w+", "", text)
    text = re.sub(r"\u0640", "", text)
    text = re.sub(r"[\u064B-\u065F\u0670]", "", text)
    text = re.sub(r"[إأآا]", "ا", text)
    text = re.sub(r"ى", "ي", text)
    text = re.sub(r"ة", "ه", text)
    text = re.sub(r"(.)\1+", r"\1\1", text)
    return re.sub(r"\s+", " ", text.replace("#", " ")).strip()

def load_data(file_path):
    df = pd.read_csv(file_path, keep_default_na=False)
    for col in ["target", "stance", "sentiment", "sarcasm", "text"]:
        df[col] = df[col].astype(str).str.strip()
        
    df["clean_text"] = df["text"].apply(clean_arabic_tweet)
    df["input_text"] = df["target"] + " [SEP] " + df["clean_text"]
    
    df["label_stance"] = df["stance"].map(STANCE2ID).fillna(2).astype(int)
    df["label_sentiment"] = df["sentiment"].map(SENTIMENT2ID).fillna(1).astype(int) 
    df["label_sarcasm"] = df["sarcasm"].map(SARCASM2ID).fillna(0).astype(int)
    return df

full_train_df = load_data("../data/train.csv")
full_dev_df = load_data("../data/dev.csv")

def tokenize_func(examples):
    tokenized = tokenizer(examples["input_text"], padding="max_length", truncation=True, max_length=128)
    tokenized["labels_stance"] = examples["label_stance"]
    tokenized["labels_sentiment"] = examples["label_sentiment"]
    tokenized["labels_sarcasm"] = examples["label_sarcasm"]
    return tokenized

# 3. KIẾN TRÚC MÔ HÌNH VÀ TRAINER (Kế thừa từ bản Tối ưu)
class MultiTaskMARBERT(nn.Module):
    def __init__(self, model_name):
        super().__init__()
        self.bert = AutoModel.from_pretrained(model_name)
        hidden_size = self.bert.config.hidden_size
        self.stance_head = nn.Linear(hidden_size, 3)
        self.sentiment_head = nn.Linear(hidden_size, 3)
        self.sarcasm_head = nn.Linear(hidden_size, 2)

    def forward(self, input_ids, attention_mask, token_type_ids=None):
        outputs = self.bert(input_ids=input_ids, attention_mask=attention_mask, token_type_ids=token_type_ids)
        pooled = outputs.pooler_output 
        return self.stance_head(pooled), self.sentiment_head(pooled), self.sarcasm_head(pooled)

class PerTargetTrainer(Trainer):
    def __init__(self, stance_weights, *args, **kwargs):
        super().__init__(*args, **kwargs)
        self.stance_weights = stance_weights # Nhận trọng số động cho từng Target

    def compute_loss(self, model, inputs, return_outputs=False, **kwargs):
        labels_stance = inputs.pop("labels_stance")
        labels_sentiment = inputs.pop("labels_sentiment")
        labels_sarcasm = inputs.pop("labels_sarcasm")
        
        logits_stance, logits_sentiment, logits_sarcasm = model(**inputs)
        
        # Label Smoothing 0.1 và Class Weights động
        loss_fct_stance = nn.CrossEntropyLoss(weight=self.stance_weights.to(model.bert.device), label_smoothing=0.1)
        loss_fct_sentiment = nn.CrossEntropyLoss()
        loss_fct_sarcasm = nn.CrossEntropyLoss()
        
        loss_stance = loss_fct_stance(logits_stance, labels_stance)
        loss_sentiment = loss_fct_sentiment(logits_sentiment, labels_sentiment)
        loss_sarcasm = loss_fct_sarcasm(logits_sarcasm, labels_sarcasm)
        
        total_loss = loss_stance + 0.2 * loss_sentiment + 0.1 * loss_sarcasm
        return (total_loss, {"logits_stance": logits_stance}) if return_outputs else total_loss

def compute_metrics(eval_pred):
    logits_tuple = eval_pred.predictions
    labels_tuple = eval_pred.label_ids

    logits = logits_tuple["logits_stance"] if isinstance(logits_tuple, dict) else (logits_tuple[0] if isinstance(logits_tuple, (tuple, list)) else logits_tuple)
    labels = labels_tuple[0] if isinstance(labels_tuple, (tuple, list)) else labels_tuple

    logits, labels = np.array(logits), np.array(labels)
    if logits.ndim > 2:
         logits = logits.reshape(-1, logits.shape[-1])
         labels = labels.flatten()
    elif logits.ndim == 1 and labels.ndim == 0:
         logits = logits.reshape(1, -1)
         labels = labels.reshape(1)

    preds = np.argmax(logits, axis=-1)
    f_against = f1_score(labels, preds, labels=[0], average="macro")
    f_favor = f1_score(labels, preds, labels=[1], average="macro")
    return {"Favg2": (f_favor + f_against) / 2.0}

# =====================================================================
# 4. VÒNG LẶP HUẤN LUYỆN PER-TARGET (BƯỚC 2)
# =====================================================================
TARGETS = ["Covid Vaccine", "Digital Transformation", "Women empowerment"]
cols_to_remove = ["input_text", "label_stance", "label_sentiment", "label_sarcasm", "target", "stance", "sentiment", "sarcasm", "text", "clean_text"]

for target in TARGETS:
    print(f"\n{'='*50}")
    print(f"🚀 ĐANG HUẤN LUYỆN CHUYÊN GIA CHO: {target}")
    print(f"{'='*50}")
    
    # 1. Lọc Data theo Target
    train_df = full_train_df[full_train_df["target"] == target].copy()
    dev_df = full_dev_df[full_dev_df["target"] == target].copy()
    
    print(f"Số lượng mẫu Train: {len(train_df)} | Dev: {len(dev_df)}")
    
    # 2. Tính lại Class Weights riêng cho Target này (Vì phân phối nhãn mỗi chủ đề rất khác nhau)
    stance_labels = train_df["label_stance"].tolist()
    stance_weights = compute_class_weight(class_weight="balanced", classes=np.unique(stance_labels), y=stance_labels)
    stance_weights_tensor = torch.tensor(stance_weights, dtype=torch.float)
    
    # 3. Tạo Dataset HuggingFace
    train_dataset = Dataset.from_pandas(train_df).map(tokenize_func, batched=True)
    dev_dataset = Dataset.from_pandas(dev_df).map(tokenize_func, batched=True)
    
    # Giữ lại các cột cần thiết cho Trainer
    train_dataset = train_dataset.remove_columns([c for c in cols_to_remove if c in train_dataset.column_names])
    dev_dataset = dev_dataset.remove_columns([c for c in cols_to_remove if c in dev_dataset.column_names])
    
    # 4. Khởi tạo Model & Tham số (Có Early Stopping vì data ít)
    model = MultiTaskMARBERT(MODEL_NAME)
    safe_target_name = target.replace(" ", "_")
    output_dir = f"../model/per_target_{safe_target_name}"
    
    training_args = TrainingArguments(
        output_dir=output_dir,
        learning_rate=2e-5,
        per_device_train_batch_size=16,
        num_train_epochs=6,
        warmup_ratio=0.1,
        lr_scheduler_type="cosine",
        weight_decay=0.01,
        eval_strategy="epoch",
        save_strategy="epoch",  
        load_best_model_at_end=False, # Tắt để custom model không lỗi, ta sẽ lưu epoch cuối
        label_names=["labels_stance", "labels_sentiment", "labels_sarcasm"],
    )
    
    trainer = PerTargetTrainer(
        stance_weights=stance_weights_tensor, # Truyền trọng số động vào
        model=model,
        args=training_args,
        train_dataset=train_dataset,
        eval_dataset=dev_dataset,
        data_collator=data_collator,
        compute_metrics=compute_metrics,
    )
    
    # 5. Huấn luyện và Lưu
    trainer.train()
    
    save_path = f"../model/best_local_{safe_target_name}.pt"
    torch.save(model.state_dict(), save_path)
    print(f"✅ Đã lưu chuyên gia {target} tại: {save_path}")
    
    # Giải phóng GPU cho vòng lặp tiếp theo
    del model, trainer
    torch.cuda.empty_cache()

print("\n🎉 HOÀN TẤT HUẤN LUYỆN 3 MÔ HÌNH PER-TARGET!")

d:\StanceEval-2026\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm



🚀 ĐANG HUẤN LUYỆN CHUYÊN GIA CHO: Covid Vaccine
Số lượng mẫu Train: 1167 | Dev: 206


Loading weights: 100%|██████████| 199/199 [00:00<00:00, 2630.41it/s]
[transformers] warmup_ratio is deprecated and will be removed in v5.2. Use `warmup_steps` instead.


Epoch,Training Loss,Validation Loss,Favg2
1,No log,1.274681,0.634804
2,No log,1.141469,0.725676
3,No log,1.159149,0.736513
4,No log,1.202361,0.762358
5,No log,1.233461,0.764793
6,No log,1.232680,0.759615


✅ Đã lưu chuyên gia Covid Vaccine tại: ../model/best_local_Covid_Vaccine.pt

🚀 ĐANG HUẤN LUYỆN CHUYÊN GIA CHO: Digital Transformation
Số lượng mẫu Train: 1145 | Dev: 203


...

Epoch,Training Loss,Validation Loss,Favg2
1,No log,1.362471,0.650145
2,No log,1.243710,0.679644
3,No log,1.178868,0.705700
4,No log,1.195569,0.709587
5,No log,1.226494,0.733644
6,No log,1.226373,0.731795


✅ Đã lưu chuyên gia Digital Transformation tại: ../model/best_local_Digital_Transformation.pt

🚀 ĐANG HUẤN LUYỆN CHUYÊN GIA CHO: Women empowerment
Số lượng mẫu Train: 1190 | Dev: 210


Loading weights: 100%|██████████| 199/199 [00:00<00:00, 1695.14it/s]
[transformers] warmup_ratio is deprecated and will be removed in v5.2. Use `warmup_steps` instead.


Epoch,Training Loss,Validation Loss,Favg2
1,No log,1.292425,0.730399
2,No log,1.170665,0.830399
3,No log,1.178056,0.807191
4,No log,1.149658,0.849848
5,No log,1.136280,0.863283
6,No log,1.134782,0.862267


✅ Đã lưu chuyên gia Women empowerment tại: ../model/best_local_Women_empowerment.pt

🎉 HOÀN TẤT HUẤN LUYỆN 3 MÔ HÌNH PER-TARGET!
